In [ ]:
import warnings

import numpy as np
from ipynb.fs.defs.ARMA import arma_residuals, fit_arma
from ipynb.fs.defs.Tail_Index import tail_index
from scipy import stats
from scipy.optimize import brentq, minimize

warnings.filterwarnings('ignore')

# Constrained Forecasting
- Find Alpha and Betas which Force a Specific Tail Index
- Optimise Likelihood over these Values

## Find Alpha From TI

In [6]:
def alpha_fn(alpha, dist, ti, beta):
    try:
        return tail_index(dist, alpha, beta) - ti
    except:
        return 10**10

In [7]:
def find_alpha(ti, dist, beta):
    alpha = np.nan
    alpha = brentq(alpha_fn, 0, 1-beta, xtol=1e-8, args=(dist, ti, beta))
    return alpha

## Constrained Forecasting

In [11]:
def negloglik_constrained(params, ret, p, q, ti, forecast=False):
    '''Negative Log-Likelihood for Given Beta and Tail Index'''
    
    # Mean Estimation
    res = fit_arma(ret, p, q)
    resid, next = arma_residuals(res, ret, p, q, True)
    fitted = ret - resid
    
    alpha_0, beta_1 = params
    alpha_1 = find_alpha(ti, stats.norm(), beta_1)
    
    # Compute Errors
    errors = ret - fitted
    
    # Initialise Variance
    vars = np.zeros(len(ret))
    vars[0] = np.var(errors)
    
    # Compute Variances
    for i in range(1, len(ret)):
        vars[i] = alpha_0 + alpha_1 * errors[i-1]**2 + beta_1 * vars[i-1]
        
    loglik = -0.5 * np.sum(np.log(2 * np.pi) + np.log(vars) + errors**2 / vars)
    
    # Forecast Next Mean and Sigma
    mu = next
    
    # Forecast Next Volatility
    sigma = alpha_0 + alpha_1 * errors[-1]**2 + beta_1 * vars[-1]
    
    if forecast:
        return -loglik, mu, np.sqrt(sigma)
    else:
        return -loglik

In [13]:
def confint(mu, sigma, confidence=0.95):
    '''Create Confidence Intervals from Mean and Volatility'''
    alpha = 1 - confidence
    z = stats.norm.ppf(1 - alpha/2)
    margin = z * (sigma)
    
    lower = mu - margin
    upper = mu + margin
    
    return lower, upper

In [12]:
def constrained_forecast(ret, init, bounds, p, q, ti):
    '''Forecast Mean and Volatility over Constrained Tail Index'''
    res = minimize(
        negloglik_constrained,
        x0=init,
        args=(ret*1000, p, q, ti),
        bounds=bounds,
        tol=10**-1
    )

    alpha_0, beta_1 = res.x
    alpha_0 /= 1000000
    alpha_1 = find_alpha(ti, stats.norm(), beta_1)

    loglik, mu, sigma = negloglik_constrained([alpha_0, beta_1], ret, p, q, ti, True)
    lower, upper = confint(mu, sigma)
    return alpha_1, beta_1, mu, sigma, lower, upper